In [1]:
%reload_ext autotime
import pandas as pd
import os
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
from pprint import pprint
import json5 as json # This is a more forgiving JSON parser that can handle comments, single quotes, and trailing commas
from glob import glob
from tqdm import tqdm
from openai import OpenAI
import base64
files = sorted(glob("../supplements_videos/*.json"))
print(len(files))

4392
time: 875 ms (started: 2026-06-09 12:55:27 +12:00)


In [2]:
from glob import glob
pd.Series([os.path.splitext(f)[1] for f in glob("../supplements_videos/*")]).value_counts()

.json    4392
.mp4     3104
.webm    1228
.mp3       52
.m4a       26
.mkv        6
Name: count, dtype: int64

time: 40.5 ms (started: 2026-06-09 12:55:28 +12:00)


In [3]:
print(pd.to_timedelta("00:" + pd.read_csv("../data/supplements.csv").duration).describe())
print("Total video time:", pd.to_timedelta("00:" + pd.read_csv("../data/supplements.csv").duration).sum())

count                      7367
mean     0 days 00:00:59.388896
std      0 days 00:00:39.888591
min             0 days 00:00:02
25%             0 days 00:00:28
50%             0 days 00:00:54
75%             0 days 00:01:21
max             0 days 00:03:00
Name: duration, dtype: object
Total video time: 5 days 01:31:58
time: 87.1 ms (started: 2026-06-09 12:55:28 +12:00)


In [4]:
pd.read_csv("../data/supplements.csv").sort_values("duration")

,link,duration,title,source,author
3416,https://www.instagram.com/reel/DY7iaRBxCcP/,0:02,Natalie Yco | Hi Fam! 🙋🏾‍♀️ Tell me your holy grail ...,Instagram,Natalie Yco
1792,https://www.instagram.com/p/DXFnVgZgcJT/,0:02,BONE HEALTH IS MY NEW OBSESSION Many with prolapse ...,Instagram,Kim Vopni - The Vagina Coach
2554,https://www.instagram.com/reel/DU2jYyyCK0L/,0:02,HAIR FOOD | MENO is a menopause specific food ...,Instagram,Oxygen Hair Beaconsfield
1365,https://www.facebook.com/mikkiwillidennutrition/posts/you-cant-get-everything-you-need-from-food...,0:03,"You can't get everything you need from food, despite what ...",Facebook,"Mikki Williden, PhD"
1132,https://www.facebook.com/emilydidonato/posts/the-secrets-out-my-favorite-vichylaboratoires-serum...,0:03,"The secret's out, my favorite @vichylaboratoires serum is now ...",Facebook,Emily DiDonato
...,...,...,...,...,...
4367,https://www.tiktok.com/@dr.anika.ackerman/video/7192289140925320491,3:00,Boost Your Sex Drive: Female Libido Treatments Explained,TikTok,dr.anika.ackerman
6801,https://www.youtube.com/shorts/cN7Y0iRtxfo,3:00,Médico Revela: O melhor horário para tomar suplementos | Dr ...,YouTube,Dr Rogério Bocardo
2877,https://www.instagram.com/reel/DWwPG4ejDXj/,3:00,Let's talk VITAMINS and perimenopause! I take both morning ...,Instagram,Lissette
6948,https://www.youtube.com/shorts/kZVpXCmJjQo,3:00,Menopause Supplements That Actually Work,YouTube,fitwithpratham


time: 39.9 ms (started: 2026-06-09 12:55:28 +12:00)


In [5]:
json_filename = files[1]
with open(json_filename) as f:
    data = json.load(f)
display(pd.json_normalize(data))
print(data["description"])

,id,formats,channel,channel_id,uploader,uploader_id,channel_url,uploader_url,track,artists,duration,title,description,timestamp,view_count,like_count,repost_count,comment_count,save_count,thumbnails,webpage_url,webpage_url_basename,webpage_url_domain,extractor,extractor_key,thumbnail,display_id,fulltitle,duration_string,upload_date,artist,epoch,ext,vcodec,acodec,format_id,tbr,quality,preference,filesize,width,height,url,protocol,video_ext,audio_ext,resolution,dynamic_range,aspect_ratio,cookies,format,_type,http_headers.User-Agent,http_headers.Accept,http_headers.Accept-Language,http_headers.Sec-Fetch-Mode,http_headers.Referer,_version.version,_version.release_git_head,_version.repository
0,7627282165595770126,"[{'ext': 'mp4', 'vcodec': 'h264', 'acodec': 'aac', 'format_id': 'h264_540p_479619-0', 'tbr': 479...",recipeguy,MS4wLjABAAAAvxF9svVYFioohDjC6ZNswV3LWTm2sJduGgezvLDv2jssNN7pBMVKCHMRhfLnmmF5,therecipeeguy,7410406742842131499,https://www.tiktok.com/@MS4wLjABAAAAvxF9svVYFioohDjC6ZNswV3LWTm2sJduGgezvLDv2jssNN7pBMVKCHMRhfLn...,https://www.tiktok.com/@therecipeeguy,original sound,[recipeguy],35,#0positiv #meno #40pluswomen #eyehealth #haitianusa,#0positiv #meno #40pluswomen #eyehealth #haitianusa,1775865045,14400,44,55,5,19,"[{'id': 'dynamicCover', 'url': 'https://p16-common-sign.tiktokcdn.com/tos-useast5-p-0068-tx/oYAc...",https://www.tiktok.com/@therecipeeguy/video/7627282165595770126,7627282165595770126,tiktok.com,TikTok,TikTok,https://p16-common-sign.tiktokcdn.com/tos-useast5-p-0068-tx/oENfYivhANdRcB6OIXjDfI8YroiE2poSgBBZ...,7627282165595770126,#0positiv #meno #40pluswomen #eyehealth #haitianusa,35,20260410,recipeguy,1780662769,mp4,h265,aac,bytevc1_1080p_917376-1,917,3,-1,4024988,1080,1920,https://v19-webapp-prime.tiktok.com/video/tos/maliva/tos-maliva-ve-0068c799-us/o8I1ZpdEoBYDngRU6...,https,mp4,none,1080x1920,SDR,0.56,ttwid=1%7Cpv9i9XP6eoVeWY5GmwowCyUeJV4Z3xsVS0JzDF_8lfg%7C1780655270%7C379da9b297c4d3d97bbdb2c653b...,bytevc1_1080p_917376-1 - 1080x1920,video,"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0....","text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8","en-us,en;q=0.5",navigate,https://www.tiktok.com/@therecipeeguy/video/7627282165595770126,2026.03.17,04d6974f502bbdfaed72c624344f262e30ad9708,yt-dlp/yt-dlp


#0positiv #meno  #40pluswomen  #eyehealth  #haitianusa 
time: 305 ms (started: 2026-06-09 12:55:28 +12:00)


In [6]:
def get_prompt(data):
    return f"""This is a video downloaded from {data['extractor']}. Here's the description of the video: {data['description']}.

        The creator of the video is {data.get('channel', 'an unknown channel')} ({data.get('uploader', 'an unknown uploader')})
        It has {data.get('like_count', 'an unknown number of')} likes, {data.get('view_count', 'an unknown number of')} views, and {data.get('comment_count', 'an unknown number of')} comments.
        Taking into account this description, and the video, extract the following information, in JSON format:
        description: What is happening in the video? Provide a detailed description of the actions, context, and any notable elements present in the video.
        transcript: If there is any spoken content in the video, transcribe it into English. If there is no spoken content, indicate "No spoken content".
        tone: What is the overall tone or mood of the video? Is it humorous, serious, educational, emotional, etc.?
        supplements: Does the video mention any supplements, vitamins, or medications? If so, list them. If not, indicate "No supplements mentioned".
        active_ingredients: If any supplements are mentioned, list the active ingredients in those supplements. If no supplements are mentioned, indicate "No active ingredients mentioned".
        symptoms: Does the video mention any specific symptoms, conditions, or health issues? If so, list them. If not, indicate "No symptoms mentioned".
        menopause: Is the video specifically targeting the supplement towards menopause-related symptoms or conditions? Answer True or False.
        language: What language is this video in?
        marketing: Is this video promoting or advertising any product, service, brand, or organization? If so, what is it? Otherwise, indicate "No marketing content".
        job: For the main speaker, what is their job or profession? If it is not mentioned in the video, indicate "No job information". A comma separated string, one or more of the following: therapist, psychologist, pediatrician, doctor, nurse, teacher, professor, social worker, counselor, coach, influencer, content creator?
        sentiment: Does this video recommend a particular supplement, discourage it, or is it neutral? One of negative, neutral or positive
        criticism: If the video is critical of a particular supplement, what are the main criticisms mentioned? 
        alternative_strategies: Does the video mention any alternative strategies to supplements? If so, what are they? A comma separated string. If no alternatives are mentioned, indicate "No alternative strategies mentioned".
        usefulness: Rate the overall usefulness of the video on a scale from 1 to 10, where 1 is not useful at all and 10 is extremely useful.
        misleading: Rate the extent to which the video contains misleading or inaccurate information on a scale from 1 to 10, where 1 is not misleading at all and 10 is extremely misleading.
        quality: Rate the overall quality of the video on a scale from 1 to 10, where 1 is very poor quality and 10 is excellent quality.
        personal_experience: Does the speaker mention any personal experience with supplements? If so, briefly summarize it.

        Do not include comments in your JSON response. Only respond with the JSON object. Make sure the JSON is valid
    """
print(get_prompt(data))

This is a video downloaded from TikTok. Here's the description of the video: #0positiv #meno  #40pluswomen  #eyehealth  #haitianusa .

        The creator of the video is recipeguy (therecipeeguy)
        It has 44 likes, 14400 views, and 5 comments.
        Taking into account this description, and the video, extract the following information, in JSON format:
        description: What is happening in the video? Provide a detailed description of the actions, context, and any notable elements present in the video.
        transcript: If there is any spoken content in the video, transcribe it into English. If there is no spoken content, indicate "No spoken content".
        tone: What is the overall tone or mood of the video? Is it humorous, serious, educational, emotional, etc.?
        supplements: Does the video mention any supplements, vitamins, or medications? If so, list them. If not, indicate "No supplements mentioned".
        active_ingredients: If any supplements are mentioned,

In [7]:
client = OpenAI(base_url="https://ai.cer-sandbox.cloud.edu.au/v1", api_key="not needed")

video_filename = json_filename.replace("info.json", data["ext"])
print(video_filename)
with open(video_filename, "rb") as video_file:
    base64_video = "data:video/" + data["ext"] + ";base64," + base64.b64encode(video_file.read()).decode("utf-8")
print(len(base64_video))
 
response = client.chat.completions.create(
    model="nemotron_3_nano_omni",
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "video_url", "video_url": {"url": base64_video}},
                {"type": "text", "text": get_prompt(data)},
            ],
        }
    ],
    max_tokens=20480,
    temperature=0.6,
    top_p=0.95,
    extra_body={
        "chat_template_kwargs": {
            "enable_thinking": False,
        },
        "mm_processor_kwargs": {"use_audio_in_video": True},
    },
)
pprint(json.loads(response.choices[0].message.content))

../supplements_videos/#0positiv #meno  #40pluswomen  #eyehealth  #haitianusa  [7627282165595770126].mp4
5366674
{'active_ingredients': 'No active ingredients mentioned',
 'alternative_strategies': 'No alternative strategies mentioned',
 'criticism': 'No criticisms mentioned',
 'description': 'A person wearing blue gloves holds a blue cylindrical '
                "container labeled 'meno' and 'EYE Health Capsules'. They open "
                'the container, pour out brown capsules into the lid, and then '
                'pick up one capsule to display it. The background features a '
                'light-colored surface with two gold geometric objects. The '
                "video focuses on showcasing the product's packaging, "
                'contents, and the capsule itself.',
 'job': 'content creator',
 'language': 'French',
 'marketing': 'meno',
 'menopause': True,
 'misleading': 4,
 'personal_experience': 'The speaker mentions having tried the product and '
                  